# 06 — MEvoLattice v22 Compatibility Export

현재 Voxel descriptor dataset을 첨부 `Com_Training_Visualization_v22.ipynb`의 구조인자 Excel 입력과 최대한 호환되는 형태로 내보냅니다.

- `Summary` sheet의 ID column은 기존 코드가 기대하는 `모델명`으로 생성합니다.
- `Type`, `Source`, `Target VF` metadata를 유지합니다.
- 구조인자 열은 Voxel descriptor 원래 이름을 그대로 둡니다.
- **주의:** 첨부 `Com_Optimization_v19`의 geometry mutation은 line/strut graph 전용이므로 Voxel 역설계에는 새 `03_Hierarchical_Inverse_Optimization_v1`을 사용해야 합니다.


In [ ]:
from pathlib import Path
import sys,pandas as pd,json
PROJECT_POINTER=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation\.ai_voxel_ml_project.json")
CODE_DIR=Path.cwd()/"Code" if (Path.cwd()/"Code").exists() else Path.cwd();sys.path.insert(0,str(CODE_DIR)) if str(CODE_DIR) not in sys.path else None
from voxel_ml_common import load_contract,load_all_structures
c=load_contract(PROJECT_POINTER);df=load_all_structures(c);out=Path(c['data_root'])/'MEvo_v22_Compatibility';out.mkdir(parents=True,exist_ok=True)
df['_rank']=df.get('meta__fidelity','screening').astype(str).map({'final':2,'screening':1}).fillna(0)
df=df.sort_values('_rank',ascending=False).drop_duplicates('meta__design_id',keep='first') if 'meta__design_id' in df else df.drop_duplicates('sample_id')
meta=pd.DataFrame({'모델명':df.get('meta__design_id',df['sample_id']).astype(str),'Type':df.get('gen__voxel_mode','voxel'),'Source':df.get('meta__run_id','AI-Voxel'),'Target VF':pd.to_numeric(df.get('gen__target_vf',float('nan')),errors='coerce')})
desc=[c0 for c0 in df.columns if not c0.startswith(('gen__','latent__','meta__')) and c0!='sample_id']
summary=pd.concat([meta.reset_index(drop=True),df[desc].reset_index(drop=True)],axis=1)
xlsx=out/'Structural_Factors_All.xlsx'
with pd.ExcelWriter(xlsx,engine='openpyxl') as w: summary.to_excel(w,sheet_name='Summary',index=False)
summary.to_csv(out/'Structural_Factors_All.csv',index=False,encoding='utf-8-sig')
print('MEvo v22 structural-factor file:',xlsx);display(summary.head())
print('Set Com_Training_Visualization_v22 STRUCTURE_VARIABLE_FILE to this xlsx.')
